# HEIST 🔍
**AI detective vs AI criminal money laundering — Multi-Agent RL**

This notebook trains an investigator agent using **GRPO (trl/unsloth)** on Colab's T4 GPU.

- 20 episodes (~30 minutes)
- Co-evolving AI criminal writes attack code into the Criminal Codex
- Zero-Day Reveal at the end confirms novelty


In [ ]:
# @title 1. Install Dependencies
!pip install -q unsloth 'trl==0.29.0' peft torch transformers accelerate bitsandbytes datasets
!pip install -q networkx numpy pydantic 'openenv-core>=0.2.1' fastapi uvicorn requests pandas matplotlib plotly

import os
import sys

REPO_URL = 'https://github.com/YOUR_USERNAME/heist'  # Update if using a remote clone
if not os.path.exists('/content/heist'):
    !git clone {REPO_URL} /content/heist
    
os.chdir('/content/heist')
sys.path.insert(0, '/content/heist')
sys.path.insert(0, '/content/heist/env')

In [ ]:
# @title 2. Set API Keys
# Required for the Criminal Designer, Compliance Expert, and Oversight Agent
os.environ['GEMINI_API_KEY'] = '' # <-- PASTE KEY HERE
os.environ['HEIST_MODEL']    = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'

print("API Keys set.")

In [ ]:
# @title 3. Generate Transaction Graph
from env.transaction_graph import TransactionGraph
from env.scenarios import ALL_SCENARIOS

print("Building transaction graph...")
graph = TransactionGraph(num_nodes=5000)
print(f"Graph generated with {graph.graph.number_of_nodes()} nodes and {graph.graph.number_of_edges()} edges.")
print(f"Loaded {len(ALL_SCENARIOS)} seed scenarios.")

In [ ]:
# @title 4. Load Environment & Run 1 Manual Episode
from env.heist_env import HeistEnvironment
from agent_heuristic import get_heuristic_action

env = HeistEnvironment()
obs = env.reset(seed=101)
print(f"\nStarted Episode: {env.state.scheme_id}")
print(f"Scheme Type: {env.state.scheme_type}")

steps = 0
while not obs.done and steps < 10:
    act = get_heuristic_action(obs.model_dump())
    obs = env.step(act)
    steps += 1
    print(f"Step {steps} | Phase: {obs.current_phase} | Tool: {act.action_type.name} | R={obs.reward:+.3f}")

In [ ]:
# @title 5. Run GRPO Training (20 Episodes)
import train.train_grpo as tg

os.environ['NUM_EPISODES'] = '20'
os.environ['GRPO_G']       = '4'
os.environ['LR']           = '5e-5'
os.environ['CODEX_K']      = '5'
os.environ['OUT_DIR']      = '/content/heist/checkpoints'

print("Starting GRPO Training on T4... This will take ~30 minutes.")
curves = tg.train(
    num_episodes=20,
    use_trl_trainer=False,
    use_model=True, 
    verbose=True
)

In [ ]:
# @title 6. Show Reward Curve
import json
import matplotlib.pyplot as plt

with open('/content/heist/training_curves.json') as f:
    curves = json.load(f)

plt.figure(figsize=(10, 4))
plt.plot(curves['episodes'], curves['r_inv'], 'b-o', markersize=5)
plt.title('R_investigator over Training')
plt.xlabel('Episode')
plt.ylabel('R_investigator (Total)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# @title 7. Show Criminal Codex Growth
with open('/content/heist/criminal_codex.py', 'r') as f:
    lines = f.readlines()

funcs = [l.strip() for l in lines if l.startswith('def inject_')]
print(f"\nCriminal Codex Total Schemes: {len(funcs)}")
print("-" * 40)
for i, fn in enumerate(funcs):
    prefix = '[🌱 Seed]' if i < 4 else '[🤖 AI-Generated]'
    print(f"{prefix} {fn.split('(')[0].replace('def ', '')}")

In [ ]:
# @title 8. Run Zero-Day Reveal
from eval.evaluate import zero_day_reveal

print("\n--- TRIGGERING ZERO DAY REVEAL ---")
zd_metrics = zero_day_reveal(n_trials=5, verbose=True)

In [ ]:
# @title 9. Show Final Comparison Table
from eval.evaluate import compare_models

print("\n--- RUNNING BASELINES VS ZERO-DAY ---")
# Note: In a real notebook, you would pass the initialized policy function here:
# compare_models(trained_policy_fn=my_qwen_policy)
metrics = compare_models(n_episodes=5, verbose=True)